In [1]:
device = "cpu"
compute_type = "float32"

SAMPLE_RATE = 16_000

In [2]:
import time
import threading
import pyaudio
from RealtimeSTT import AudioToTextRecorder

def find_input_device_index(name_substring: str) -> int:
    """
    Look up a PyAudio input device index by (partial) name.

    Args:
        name_substring: case-insensitive substring to match, e.g. "MacBook Pro Microphone".

    Returns:
        Device index to pass as input_device_index.
    """
    p = pyaudio.PyAudio()
    try:
        for i in range(p.get_device_count()):
            info = p.get_device_info_by_index(i)
            if info["maxInputChannels"] > 0 and name_substring.lower() in info["name"].lower():
                return i
    finally:
        p.terminate()
    raise ValueError(f"no input device matching {name_substring!r}")

MACBOOK_MIC_INDEX = find_input_device_index("MacBook Pro Microphone")
print(f"default input mic: MacBook Pro Microphone (device index {MACBOOK_MIC_INDEX})")

default input mic: MacBook Pro Microphone (device index 3)


In [3]:
# Wake-word gating: mic listens idle, only transcribes after wake word detected.
# Using openWakeWord's pretrained "hey_jarvis" model as a placeholder for a
# future custom-trained "Hey Ordo" model (swap via WAKE_WORD_MODEL_PATH/WAKE_WORD).
import os
import numpy as np
import openwakeword
openwakeword.utils.download_models()  # no-op if already cached

# NOTE: openwakeword's Model() loads ALL bundled pretrained models when no
# explicit path is given (alexa, hey_mycroft, etc), so it would also fire on
# words other than "hey_jarvis". We pass an explicit model path so only the
# one wake word is loaded/active. Swap this path to a custom "Hey Ordo" model
# (.onnx/.tflite) once trained -- no other code changes needed.
WAKE_WORD = "hey_jarvis"
WAKE_WORD_MODEL_PATH = os.path.join(
    os.path.dirname(openwakeword.__file__), "resources", "models", "hey_jarvis_v0.1.onnx"
)

# Clipping check: int16 PCM samples pinned at +/-32767 indicate the mic input
# gain was too hot for the signal, producing flat-topped waveforms. Whisper
# transcribes clipped audio fluently but confidently wrong, so we only warn
# (not reject) -- a human can still judge the printed text against this flag.
INT16_CLIP_VALUE = 32767
CLIP_RATIO_WARN_THRESHOLD = 0.001  # warn if >0.1% of samples in an utterance are clipped

def clipped_sample_ratio(audio_int16: np.ndarray) -> float:
    """
    Fraction of samples sitting at the int16 clipping ceiling.

    Args:
        audio_int16: 1D array of int16 PCM samples for one utterance.

    Returns:
        Ratio in [0, 1] of samples at or beyond the clip threshold.
    """
    if audio_int16.size == 0:
        return 0.0
    return float(np.mean(np.abs(audio_int16) >= INT16_CLIP_VALUE))

def make_wakeword_recorder(
    language: str | None = None,
    model_size: str = "large-v3-turbo",
    on_realtime_update=None,
    on_stabilized=None,
    on_wakeword_detected=None,
    on_recorded_chunk=None,
    input_device_index: int | None = MACBOOK_MIC_INDEX,
) -> AudioToTextRecorder:
    """
    Like make_recorder, but the recorder stays idle (no transcription) until
    the wake word is detected, then activates for one utterance, like "Hey Siri".

    Args:
        on_wakeword_detected: callback() fired the instant the wake word fires.
        on_recorded_chunk: callback(bytes) fired for every raw audio chunk
                            captured (used to accumulate samples for the
                            clipping check; fires continuously, not just
                            while actively recording an utterance).
        (other args same as make_recorder)
    """
    return AudioToTextRecorder(
        model=model_size,
        language=language or "",
        device="cpu",
        compute_type=compute_type,
        sample_rate=SAMPLE_RATE,
        input_device_index=input_device_index,
        enable_realtime_transcription=on_realtime_update is not None,
        realtime_model_type=model_size,
        realtime_processing_pause=0.2,  # how often whisper re-runs the partial pass
        on_realtime_transcription_update=on_realtime_update,
        on_realtime_transcription_stabilized=on_stabilized,
        wakeword_backend="oww",
        openwakeword_model_paths=WAKE_WORD_MODEL_PATH,
        wake_words_sensitivity=0.6,
        wake_word_activation_delay=0.0,   # require wake word immediately (0 = always required)
        wake_word_timeout=5.0,            # seconds to start speaking after wake word before re-arming
        on_wakeword_detected=on_wakeword_detected,
        on_recorded_chunk=on_recorded_chunk,
        spinner=False,
    )

def stream_transcribe_with_wakeword(
    stop_event: threading.Event,
    language: str | None = None,
    output=None,
    debug_log: list | None = None,
    max_partial_interval_s: float = 5.0,
):
    """
    Same loop as stream_transcribe, but gated behind the "Hey Jarvis" (placeholder
    for "Hey Ordo") wake word: stays silent until woken. Once woken, prints live
    partial transcript as the user speaks (on_realtime_update) -- at least once
    every `max_partial_interval_s` seconds even if the text hasn't changed, so
    the user gets regular feedback that the mic is still listening -- then prints
    the finalized utterance once it stabilizes, then goes back to listening for
    the wake word. Also tracks clipping on the captured utterance audio and warns
    if the signal was too hot.
    """
    def emit(msg: str):
        if output is not None:
            with output:
                print(msg)
        else:
            print(msg)

    start_time = time.time()
    utterance_idx = 0
    last_partial = None
    last_partial_emit_time = 0.0
    utterance_chunks: list[bytes] = []
    # Own flag instead of reading recorder.is_recording: on_recorded_chunk can
    # fire on the recording worker thread before AudioToTextRecorder's
    # constructor (called below) has returned and bound the `recorder` name,
    # so closing over `recorder` directly raced and threw a NameError.
    is_capturing = False

    def on_wake():
        nonlocal is_capturing
        is_capturing = True
        elapsed = time.time() - start_time
        emit(f"[{elapsed:6.1f}s] wake word detected, listening...")

    def on_realtime_update(text: str):
        nonlocal last_partial, last_partial_emit_time
        text = text.strip()
        if not text:
            return
        now = time.time()
        text_changed = text != last_partial
        interval_elapsed = (now - last_partial_emit_time) >= max_partial_interval_s
        if not (text_changed or interval_elapsed):
            return
        last_partial = text
        last_partial_emit_time = now
        elapsed = now - start_time
        emit(f"[{elapsed:6.1f}s] ...{text}")

    def on_recorded_chunk(chunk: bytes):
        # Only accumulate while actively recording an utterance (this fires
        # continuously, including while idly listening for the wake word).
        if is_capturing:
            utterance_chunks.append(chunk)

    def on_final(text: str):
        nonlocal utterance_idx, last_partial, last_partial_emit_time, is_capturing
        last_partial = None
        last_partial_emit_time = 0.0
        is_capturing = False

        if utterance_chunks:
            audio = np.frombuffer(b"".join(utterance_chunks), dtype=np.int16)
            utterance_chunks.clear()
            clip_ratio = clipped_sample_ratio(audio)
            if clip_ratio > CLIP_RATIO_WARN_THRESHOLD:
                emit(f"  [warning] {clip_ratio:.2%} of samples clipped -- mic gain likely too hot, transcript may be unreliable")

        if not text.strip():
            emit(f"listening for wake word ({WAKE_WORD})...")
            return
        utterance_idx += 1
        elapsed = time.time() - start_time
        emit(f"[utterance {utterance_idx:03d} | {elapsed:6.1f}s] {text}")
        if debug_log is not None:
            debug_log.append(dict(idx=utterance_idx, elapsed_s=elapsed, text=text))
        emit(f"listening for wake word ({WAKE_WORD})...")

    recorder = make_wakeword_recorder(
        language=language,
        on_wakeword_detected=on_wake,
        on_realtime_update=on_realtime_update,
        on_recorded_chunk=on_recorded_chunk,
    )

    def transcription_loop():
        while not stop_event.is_set():
            recorder.text(on_final)

    emit(f"using mic: MacBook Pro Microphone (device index {MACBOOK_MIC_INDEX})")
    emit(f"listening for wake word ({WAKE_WORD})...")
    t = threading.Thread(target=transcription_loop, daemon=True)
    t.start()

    try:
        while not stop_event.is_set():
            time.sleep(0.1)
    finally:
        recorder.shutdown()
        t.join(timeout=5)

In [4]:
wakeword_debug_log = []
wakeword_stop_event = threading.Event()
try:
    stream_transcribe_with_wakeword(stop_event=wakeword_stop_event, language="id", debug_log=wakeword_debug_log)
except KeyboardInterrupt:
    wakeword_stop_event.set()

using mic: MacBook Pro Microphone (device index 3)
listening for wake word (hey_jarvis)...
RealtimeSTT shutting down


In [ ]:
import json

TRANSCRIPTS_PATH = os.path.join("data", "transcripts.jsonl")
os.makedirs(os.path.dirname(TRANSCRIPTS_PATH), exist_ok=True)

with open(TRANSCRIPTS_PATH, "a", encoding="utf-8") as f:
    for entry in wakeword_debug_log:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"dumped {len(wakeword_debug_log)} utterances to {TRANSCRIPTS_PATH}")